# Kaggle Preprocessing Notebook\n\n**Amaç:** 2000 ham videoyu uçtan uca işleyip eğitime hazır `(30, 69)` sekans dosyalarına dönüştürmek.\n\n### Kullanım:\n1. Kaggle → Create → New Notebook → Accelerator: **GPU T4 x2**\n2. Add Data → `real life violence situations dataset`\n3. Hücreleri sırasıyla çalıştırın\n4. Output sekmesinden `.npy` dosyalarını indirin\n\n### Hücre Haritası:\n| # | Ne Yapar |\n|---|---|\n| 1 | `preprocessing.py` modülünü Kaggle'a yazar |\n| 2 | Kütüphane import + sabitler |\n| 3 | Dataset klasörlerini keşfeder |\n| 4 | Stratified split (70/15/15) |\n| 5 | Tüm videoları işler → per-video `.npy` |\n| 6 | Rastgele 5 `.npy` doğrulama |\n| 7 | Sliding window + motion filter → sekanslar |\n| 8 | Final doğrulama + özet |

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 1: preprocessing.py modülünü Kaggle ortamına yaz\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Bu hücre, tüm preprocessing fonksiyonlarını içeren modülü\n# /kaggle/working/preprocessing.py olarak diske yazar.\n# Sonraki hücrelerde bu modülden import yapılacak.\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n!pip install ultralytics -q\n\nMODULE_CODE = '''\nimport os, math, cv2\nimport numpy as np\n\n# ── Constants ─────────────────────────────────────────────\nTARGET_FPS = 10\nCLAHE_CLIP_LIMIT = 2.0\nCLAHE_TILE_GRID_SIZE = (8, 8)\nCLAHE_BRIGHTNESS_THRESHOLD = 50\nGAUSSIAN_KERNEL_SIZE = (3, 3)\nGAUSSIAN_SIGMA = 0\nRESIZE_DIM = (640, 640)\nNUM_KEYPOINTS = 17\nKEYPOINT_CONFIDENCE_THRESHOLD = 0.5\nFEATURE_DIM = 69\nSKELETON_DIM = 34\nTORSO_HEIGHT_EPSILON = 1e-6\nSEQUENCE_LENGTH = 30\nSLIDING_WINDOW_STRIDE = 15\nMOTION_FILTER_THETA = 0.05\n\n\ndef check_low_brightness(frame, threshold=CLAHE_BRIGHTNESS_THRESHOLD):\n    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n    return gray.mean() < threshold\n\n\ndef apply_clahe(frame):\n    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)\n    l_ch, a_ch, b_ch = cv2.split(lab)\n    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE)\n    l_ch = clahe.apply(l_ch)\n    lab = cv2.merge([l_ch, a_ch, b_ch])\n    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)\n\n\ndef preprocess_frame(frame):\n    if check_low_brightness(frame):\n        frame = apply_clahe(frame)\n    frame = cv2.GaussianBlur(frame, GAUSSIAN_KERNEL_SIZE, GAUSSIAN_SIGMA)\n    frame = cv2.resize(frame, RESIZE_DIM)\n    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n    return frame\n\n\ndef extract_pose(frame_rgb, model):\n    results = model(frame_rgb, verbose=False)\n    persons = []\n    if results[0].keypoints is not None and len(results[0].keypoints) > 0:\n        kps_data = results[0].keypoints.data.cpu().numpy()\n        boxes_data = results[0].boxes.data.cpu().numpy()\n        for i in range(len(kps_data)):\n            kps = kps_data[i]\n            box = boxes_data[i]\n            persons.append({\n                \"keypoints\": kps,\n                \"bbox\": box[:4],\n                \"bbox_area\": (box[2]-box[0])*(box[3]-box[1]),\n                \"bbox_center\": ((box[0]+box[2])/2, (box[1]+box[3])/2),\n                \"detection_conf\": box[4],\n            })\n    return persons\n\n\ndef select_top2_persons(persons):\n    if len(persons) == 0: return None, None\n    if len(persons) == 1: return persons[0], None\n    top2 = sorted(persons, key=lambda p: p[\"bbox_area\"], reverse=True)[:2]\n    top2 = sorted(top2, key=lambda p: p[\"bbox_center\"][0])\n    return top2[0], top2[1]\n\n\ndef filter_keypoints(keypoints):\n    filtered = np.zeros((NUM_KEYPOINTS, 2), dtype=np.float32)\n    for i in range(NUM_KEYPOINTS):\n        if keypoints[i, 2] >= KEYPOINT_CONFIDENCE_THRESHOLD:\n            filtered[i, 0] = keypoints[i, 0]\n            filtered[i, 1] = keypoints[i, 1]\n    return filtered\n\n\ndef normalize_skeleton(kps_2d):\n    hip_mx = (kps_2d[11,0] + kps_2d[12,0]) / 2\n    hip_my = (kps_2d[11,1] + kps_2d[12,1]) / 2\n    for i in range(NUM_KEYPOINTS):\n        if kps_2d[i,0] == 0.0 and kps_2d[i,1] == 0.0:\n            kps_2d[i,0] = hip_mx\n            kps_2d[i,1] = hip_my\n    centered = kps_2d.copy()\n    centered[:,0] -= hip_mx\n    centered[:,1] -= hip_my\n    sh_my = (kps_2d[5,1] + kps_2d[6,1]) / 2\n    torso_h = abs(hip_my - sh_my)\n    if torso_h > TORSO_HEIGHT_EPSILON:\n        centered /= torso_h\n    else:\n        return np.zeros((NUM_KEYPOINTS, 2), dtype=np.float32)\n    return centered\n\n\ndef build_feature_vector(person1, person2, frame_width):\n    sk1 = normalize_skeleton(filter_keypoints(person1[\"keypoints\"])).flatten() if person1 else np.zeros(SKELETON_DIM, dtype=np.float32)\n    sk2 = normalize_skeleton(filter_keypoints(person2[\"keypoints\"])).flatten() if person2 else np.zeros(SKELETON_DIM, dtype=np.float32)\n    if person1 and person2:\n        nd = math.dist(person1[\"bbox_center\"], person2[\"bbox_center\"]) / frame_width if frame_width > 0 else 1.0\n    elif person1:\n        nd = 1.0\n    else:\n        nd = 0.0\n    return np.concatenate([sk1, sk2, np.array([nd], dtype=np.float32)])\n\n\ndef process_single_video(video_path, model):\n    cap = cv2.VideoCapture(video_path)\n    if not cap.isOpened(): return None, \"Video acilamadi\"\n    native_fps = cap.get(cv2.CAP_PROP_FPS)\n    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    if native_fps <= 0 or total_frames <= 0:\n        cap.release()\n        return None, f\"Gecersiz FPS={native_fps}\"\n    interval = max(1, round(native_fps / TARGET_FPS))\n    fvecs = []\n    idx = 0\n    while True:\n        ret, frame = cap.read()\n        if not ret: break\n        if idx % interval == 0:\n            pre = preprocess_frame(frame)\n            persons = extract_pose(pre, model)\n            p1, p2 = select_top2_persons(persons)\n            fvecs.append(build_feature_vector(p1, p2, RESIZE_DIM[0]))\n        idx += 1\n    cap.release()\n    if len(fvecs) == 0: return None, \"Hic frame islenemedi\"\n    return np.array(fvecs, dtype=np.float32), None\n\n\ndef create_sliding_windows(features, window_size=SEQUENCE_LENGTH, stride=SLIDING_WINDOW_STRIDE):\n    n = features.shape[0]\n    if n < window_size:\n        padded = np.zeros((window_size, FEATURE_DIM), dtype=np.float32)\n        padded[:n] = features\n        return [padded]\n    return [features[s:s+window_size] for s in range(0, n-window_size+1, stride)]\n\n\ndef compute_motion_score(window):\n    return np.linalg.norm(np.diff(window, axis=0), axis=1).mean()\n\n\ndef apply_motion_filter(windows, theta=MOTION_FILTER_THETA):\n    return [w for w in windows if compute_motion_score(w) >= theta]\n'''\n\nwith open('/kaggle/working/preprocessing.py', 'w') as f:\n    f.write(MODULE_CODE)\n\nprint(\"preprocessing.py yazildi: /kaggle/working/preprocessing.py\")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 2: Import ve Sabitler\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# preprocessing.py'deki tüm fonksiyonları import eder.\n# Notebook'a özel sabitler (path, split oranları) burada tanımlanır.\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nimport os\nimport random\nimport numpy as np\nimport pandas as pd\nfrom glob import glob\nfrom tqdm import tqdm\nfrom ultralytics import YOLO\nfrom sklearn.model_selection import train_test_split\n\nimport sys\nsys.path.insert(0, '/kaggle/working')\nfrom preprocessing import (\n    process_single_video, create_sliding_windows,\n    apply_motion_filter, FEATURE_DIM, SEQUENCE_LENGTH\n)\n\n# Notebook-only constants\nPOSE_MODEL_NAME = \"yolov8n-pose.pt\"\nKAGGLE_INPUT = \"/kaggle/input/real-life-violence-situations-dataset\"\nOUTPUT = \"/kaggle/working\"\nVIOLENCE_LABEL = 1\nNONVIOLENCE_LABEL = 0\nSPLIT_RANDOM_STATE = 42\nTRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15\n\n# YOLO model yükle\npose_model = YOLO(POSE_MODEL_NAME)\n\nprint(\"Import tamamlandi. YOLO modeli yuklendi.\")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 3: Dataset Keşfi\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Kaggle input dizininde Violence ve NonViolence klasörlerini bulur.\n# Video sayılarını doğrular (beklenen: 1000+1000=2000).\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nfor root, dirs, files in os.walk(KAGGLE_INPUT):\n    if \"Violence\" in dirs or \"NonViolence\" in dirs:\n        VIOLENCE_DIR = os.path.join(root, \"Violence\")\n        NONVIOLENCE_DIR = os.path.join(root, \"NonViolence\")\n        break\n\nviolence_videos = sorted(glob(os.path.join(VIOLENCE_DIR, \"*\")))\nnonviolence_videos = sorted(glob(os.path.join(NONVIOLENCE_DIR, \"*\")))\n\nprint(f\"Violence:    {len(violence_videos)} video  ->  {VIOLENCE_DIR}\")\nprint(f\"NonViolence: {len(nonviolence_videos)} video  ->  {NONVIOLENCE_DIR}\")\nprint(f\"Toplam:      {len(violence_videos) + len(nonviolence_videos)}\")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 4: Stratified Split (70 / 15 / 15)\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Videoları train/val/test'e böler (sınıf oranı korunur).\n# split_map dict'i oluşturur: {video_path: (split_name, label)}\n# CSV'leri output dizinine kaydeder.\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nall_videos = violence_videos + nonviolence_videos\nall_labels = [VIOLENCE_LABEL]*len(violence_videos) + [NONVIOLENCE_LABEL]*len(nonviolence_videos)\n\ntrain_val_f, test_f, train_val_l, test_l = train_test_split(\n    all_videos, all_labels, test_size=TEST_RATIO,\n    stratify=all_labels, random_state=SPLIT_RANDOM_STATE)\n\nval_adj = VAL_RATIO / (1 - TEST_RATIO)\ntrain_f, val_f, train_l, val_l = train_test_split(\n    train_val_f, train_val_l, test_size=val_adj,\n    stratify=train_val_l, random_state=SPLIT_RANDOM_STATE)\n\nsplit_map = {}\nfor f, l in zip(train_f, train_l): split_map[f] = ('train', l)\nfor f, l in zip(val_f, val_l):     split_map[f] = ('val', l)\nfor f, l in zip(test_f, test_l):   split_map[f] = ('test', l)\n\nfor name, files, labels in [('train',train_f,train_l),('val',val_f,val_l),('test',test_f,test_l)]:\n    v = sum(1 for l in labels if l==1)\n    nv = len(labels) - v\n    print(f\"{name:5s}: {len(files):4d} video  (V={v}, NV={nv})\")\n    pd.DataFrame({'filepath':files,'label':labels,'split':name}).to_csv(\n        os.path.join(OUTPUT, f\"{name}.csv\"), index=False)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 5: Tüm Videoları İşle → Per-Video .npy\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Her videoyu process_single_video() ile işler:\n#   FPS sampling (10fps) → preprocess → YOLOv8n-Pose → multi-person\n#   → normalize → 69-dim feature vector → .npy kaydet\n# Hata olan videolar loglanır ve atlanır.\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nfor split in ['train', 'val', 'test']:\n    for label in ['violence', 'nonviolence']:\n        os.makedirs(os.path.join(OUTPUT, 'features', split, label), exist_ok=True)\n\nprocess_log = []\nskipped_videos = []\n\nfor video_path in tqdm(all_videos, desc=\"Video Isleme\"):\n    split_name, label = split_map[video_path]\n    video_name = os.path.splitext(os.path.basename(video_path))[0]\n    label_str = 'violence' if label == VIOLENCE_LABEL else 'nonviolence'\n    npy_path = os.path.join(OUTPUT, 'features', split_name, label_str, f\"{video_name}.npy\")\n\n    features, error = process_single_video(video_path, pose_model)\n\n    if features is None:\n        skipped_videos.append({'video': video_path, 'error': error})\n        continue\n\n    np.save(npy_path, features)\n    process_log.append({'video': video_name, 'split': split_name,\n                        'label': label_str, 'frames': features.shape[0]})\n\nprint(f\"\\nIslenen: {len(process_log)} | Atlanan: {len(skipped_videos)}\")\nif skipped_videos:\n    for s in skipped_videos[:5]:\n        print(f\"  SKIP: {s['video']}: {s['error']}\")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 6: Doğrulama — Rastgele 5 .npy Kontrol\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Üretilen .npy dosyalarının shape, dtype, NaN/Inf ve değer\n# aralığını kontrol eder. Sorun varsa burada yakalanır.\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nnpy_files = glob(os.path.join(OUTPUT, 'features', '**', '*.npy'), recursive=True)\nprint(f\"Toplam .npy: {len(npy_files)}\")\n\nfor f in random.sample(npy_files, min(5, len(npy_files))):\n    arr = np.load(f)\n    print(f\"  {os.path.basename(f):40s} shape={str(arr.shape):12s} \"\n          f\"min={arr.min():.2f}  max={arr.max():.2f}  \"\n          f\"NaN={np.isnan(arr).any()}  Inf={np.isinf(arr).any()}\")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 7: Sliding Window + Motion Filter → Sekanslar\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Her per-video .npy'den 30-frame'lik pencereler oluşturur.\n# Violence pencerelerine motion filter (θ=0.05) uygulanır;\n# NonViolence filtresiz kalır.\n# Train seti için NonViolence undersampling yapılır.\n# Final: X_{split}.npy (N,30,69) + y_{split}.npy (N,)\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nfor split in ['train', 'val', 'test']:\n    os.makedirs(os.path.join(OUTPUT, 'sequences', split), exist_ok=True)\n\n    v_seqs, nv_seqs = [], []\n\n    for f in glob(os.path.join(OUTPUT, 'features', split, 'violence', '*.npy')):\n        windows = create_sliding_windows(np.load(f))\n        v_seqs.extend(apply_motion_filter(windows))\n\n    for f in glob(os.path.join(OUTPUT, 'features', split, 'nonviolence', '*.npy')):\n        nv_seqs.extend(create_sliding_windows(np.load(f)))\n\n    # Train: undersampling for class balance\n    if split == 'train' and len(nv_seqs) > len(v_seqs):\n        random.seed(42)\n        nv_seqs = random.sample(nv_seqs, len(v_seqs))\n\n    X = np.array(v_seqs + nv_seqs, dtype=np.float32)\n    y = np.array([1]*len(v_seqs) + [0]*len(nv_seqs), dtype=np.float32)\n\n    np.save(os.path.join(OUTPUT, 'sequences', split, f'X_{split}.npy'), X)\n    np.save(os.path.join(OUTPUT, 'sequences', split, f'y_{split}.npy'), y)\n\n    print(f\"{split:5s}: V={len(v_seqs):5d}  NV={len(nv_seqs):5d}  \"\n          f\"Total={len(X):5d}  X={X.shape}\")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# HÜCRE 8: Final Doğrulama + İndirme Özeti\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n# Her split için X/y shape, NaN/Inf ve sınıf dağılımını kontrol\n# eder. Başarıyla tamamlanırsa indirilecek dosyaları listeler.\n# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\nprint(\"=\" * 55)\nfor split in ['train', 'val', 'test']:\n    X = np.load(os.path.join(OUTPUT, 'sequences', split, f'X_{split}.npy'))\n    y = np.load(os.path.join(OUTPUT, 'sequences', split, f'y_{split}.npy'))\n    print(f\"\\n{split.upper()}\")\n    print(f\"  X={X.shape}  y={y.shape}\")\n    print(f\"  V={int((y==1).sum())}  NV={int((y==0).sum())}\")\n    print(f\"  NaN={np.isnan(X).any()}  Inf={np.isinf(X).any()}\")\n    print(f\"  min={X.min():.3f}  max={X.max():.3f}\")\n\n# Log kaydet\nif process_log:\n    pd.DataFrame(process_log).to_csv(os.path.join(OUTPUT, 'preprocessing_log.csv'), index=False)\nif skipped_videos:\n    pd.DataFrame(skipped_videos).to_csv(os.path.join(OUTPUT, 'skipped_videos.csv'), index=False)\n\nprint(\"\\n\" + \"=\" * 55)\nprint(\"Indirilecek dosyalar (Output sekmesi):\")\nfor split in ['train', 'val', 'test']:\n    for fname in [f'X_{split}.npy', f'y_{split}.npy']:\n        p = os.path.join(OUTPUT, 'sequences', split, fname)\n        mb = os.path.getsize(p) / 1024**2\n        print(f\"  sequences/{split}/{fname}  ({mb:.1f} MB)\")\nprint(\"  train.csv / val.csv / test.csv\")\nprint(\"\\nPreprocessing tamamlandi!\")

In [ ]:
# TODO: Delete this cell — no longer needed (merged into earlier cells)

In [ ]:
# TODO: Delete this cell — no longer needed (merged into earlier cells)

In [ ]:
# TODO: Delete this cell — no longer needed (merged into earlier cells)

In [ ]:
# TODO: Delete this cell — no longer needed (merged into earlier cells)

In [ ]:
# TODO: Delete this cell — no longer needed (merged into earlier cells)